# 03 — Content-Based Filtering (Cosine Similarity)

**Tujuan:** Mendemonstrasikan inti dari sistem rekomendasi: menghitung cosine similarity antara user profile vector dan setiap job-skill vector, lalu mengurutkan untuk mendapatkan top-N rekomendasi.

**Formula Cosine Similarity:**
$$\text{sim}(u, j) = \frac{u \cdot j}{||u|| \cdot ||j||}$$

Dimana $u$ = user vector, $j$ = job vector. Nilai antara 0 (tidak mirip) sampai 1 (identik).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_jobs_data
from src.feature_engineering import build_skill_vocabulary, build_job_skill_matrix, build_user_vector
from src.recommender import compute_similarity, apply_preference_filter, get_top_recommendations

sns.set_style('whitegrid')
LI_BLUE = '#0A66C2'

## 3.1 Setup Data dan Build Vocabulary

In [ ]:
df, _ = load_jobs_data()
vocab = build_skill_vocabulary(df, skill_col='skills_list', min_freq=5)
job_matrix = build_job_skill_matrix(df, vocab, skill_col='skills_list')
print(f'Jobs: {len(df):,} | Vocabulary: {len(vocab)} skills')

## 3.2 Persona 1 — Junior Data Scientist Aspirant

In [ ]:
persona_1_skills = ['Python', 'SQL', 'Statistics', 'Pandas', 'Scikit-learn', 'Machine Learning']
user_vec_1 = build_user_vector(persona_1_skills, vocab)
scores_1 = compute_similarity(user_vec_1, job_matrix)

print(f'Persona skills: {persona_1_skills}')
print(f'\nSimilarity score statistics:')
print(f'  Mean: {scores_1.mean():.3f}')
print(f'  Max:  {scores_1.max():.3f}')
print(f'  Min:  {scores_1.min():.3f}')
print(f'  Std:  {scores_1.std():.3f}')

### Distribusi similarity score

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores_1, bins=50, color=LI_BLUE, alpha=0.85)
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('# jobs')
ax.set_title('Distribusi Similarity Score — Persona 1 (Junior Data Scientist)')
ax.axvline(scores_1.mean(), color='red', linestyle='--', label=f'Mean: {scores_1.mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()

### Top 10 rekomendasi untuk Persona 1

In [ ]:
top_1 = get_top_recommendations(df, scores_1, top_n=10)
cols_to_show = ['title', 'company_name', 'experience_level', 'med_salary', 'match_score']
cols_to_show = [c for c in cols_to_show if c in top_1.columns]
top_1[cols_to_show]

## 3.3 Persona 2 — Senior Cloud Engineer

In [ ]:
persona_2_skills = ['AWS', 'Kubernetes', 'Docker', 'Terraform', 'Python', 'Linux', 'CI/CD']
user_vec_2 = build_user_vector(persona_2_skills, vocab)
scores_2 = compute_similarity(user_vec_2, job_matrix)
top_2 = get_top_recommendations(df, scores_2, top_n=10)

print(f'Persona 2 skills: {persona_2_skills}')
top_2[cols_to_show]

### Persona 1 vs Persona 2 — Score distribution comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores_1, bins=50, alpha=0.6, label='Persona 1 (Data Scientist)', color='#0A66C2')
ax.hist(scores_2, bins=50, alpha=0.6, label='Persona 2 (Cloud Engineer)', color='#057642')
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('# jobs')
ax.set_title('Different user profiles → different job similarity distributions')
ax.legend()
plt.tight_layout()
plt.show()

## 3.4 Effect of Preference Filter

Selain similarity, kita terapkan soft penalty untuk job yang tidak match preference user (experience level, work type, salary).

In [ ]:
preferences = {
    'experience_level': 'Entry',
    'work_type': 'Remote',
    'min_salary': 60000,
    'max_salary': 100000,
}
scores_1_filtered = apply_preference_filter(df, scores_1.copy(), preferences)
top_1_filtered = get_top_recommendations(df, scores_1_filtered, top_n=10)

print('Top 10 with preferences applied (Entry / Remote / $60-100k):')
top_1_filtered[cols_to_show]

In [ ]:
# Bandingkan score sebelum vs sesudah filter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(scores_1, bins=50, color=LI_BLUE)
axes[0].set_title('Before preference filter')
axes[0].set_xlabel('Score')
axes[1].hist(scores_1_filtered, bins=50, color='#057642')
axes[1].set_title('After preference filter')
axes[1].set_xlabel('Score (with penalty)')
plt.tight_layout()
plt.show()

## Kesimpulan CBF

- Cosine similarity menghasilkan ranking yang berbeda untuk persona berbeda — sistem benar-benar memetakan user profile ke item features
- Distribusi similarity menyerupai distribusi normal yang skewed ke kiri, menunjukkan sebagian besar job tidak terlalu mirip dengan user profile manapun (sebagaimana diharapkan)
- Preference filter berperan sebagai post-processing yang menurunkan rank job yang tidak match — bukan menghilangkannya sama sekali, jadi user tetap bisa melihat alternatif